# Hysteresis test (L=4) — first vs second order

Two directed NQS sweeps per line: **forward** (warm-started up from the topological phase) and **backward** (down from the polarized phase). Read against `notes/hysteresis_schematic.png`:

* **First order** → branches *separate*: a **loop** over a coexistence window.
* **Second order** → forward and backward **coincide**: no loop.

Per sweep we plot hysteresis for the two observables that actually move, plus the energy. The movers are set by which stabilizer the field anticommutes with:

| sweep | field | magnetization | topological mover |
|---|---|---|---|
| hz | $\sigma^z$ | $\langle Z\rangle=\langle M_z\rangle$ (rises) | $\langle A_v\rangle$ (falls) |
| hx | $\sigma^x$ | $\langle X\rangle=\langle M_x\rangle$ (rises) | $\langle B_p\rangle$ (falls) |

Data: `nersc/submit_hysteresis.sh` → `check_convergence.py --dump` per branch.

In [ ]:
# ====================== 1 · CONFIG ======================
import json, os
import numpy as np
import matplotlib.pyplot as plt

ROOT = "/Users/sanzhar123/Desktop/Approximate-Symmetries-TC-main/results"
L    = 4

EXP = {
    "hz-sweep  (hx=0)":   dict(field="hz", order="2nd?",
        fwd=f"{ROOT}/hyst_L{L}_hz/forward.json",  bwd=f"{ROOT}/hyst_L{L}_hz/backward.json"),
    "hx-sweep  (hz=0)":   dict(field="hx", order="1st?",
        fwd=f"{ROOT}/hyst_L{L}_hx/forward.json",  bwd=f"{ROOT}/hyst_L{L}_hx/backward.json"),
}

# the two movers per sweep: (json key, y-label). Field anticommutes with A_v (hz) / B_p (hx).
MOVERS = {
    "hz": [("mz",  r"$\langle Z\rangle$"), ("A_v", r"$\langle A_v\rangle$  (topological)")],
    "hx": [("mx",  r"$\langle X\rangle$"), ("B_p", r"$\langle B_p\rangle$  (topological)")],
}
N_EDGES  = lambda L: 3*L**2*(L-1)   # =144 at L=4
LOOP_TOL = 0.05                     # |fwd-bwd| above this (in the row-0 mover) = 'in the loop'
BLUE, RED = "#1f77b4", "#d62728"
print("experiments:", list(EXP))

## 2 · Load the two branches

In [ ]:
# ====================== 2 · DATA ======================
def load_branch(path):
    """A check_convergence --dump json -> dict of sorted float arrays. None if absent.
    Null entries (in-flight ok(ckpt) rows with no observables yet) -> np.nan."""
    if not os.path.exists(path): return None
    d = json.load(open(path))
    h = np.array(d["field"], float); o = np.argsort(h)
    def arr(k):
        v = d.get(k)
        if v is None: return None
        return np.array([np.nan if x is None else x for x in v], float)[o]
    out = {"h": h[o], "L": int(d.get("L", L))}
    for k in ("E", "mz", "mx", "A_v", "B_p", "Vscore"):
        out[k] = arr(k)
    return out

DATA = {}
for name, e in EXP.items():
    fwd, bwd = load_branch(e["fwd"]), load_branch(e["bwd"])
    DATA[name] = dict(e, fwd=fwd, bwd=bwd)
    def _n(b): return 'MISSING' if b is None else f"{np.isfinite(b['h']).sum()} pts [{b['h'].min():.3g},{b['h'].max():.3g}]"
    print(f"[{name}]  forward={_n(fwd)}   backward={_n(bwd)}")

## 3 · Hysteresis per observable + loop readout

Rows = the two movers for that sweep, then $E/N$. Blue = forward, red = backward. A gap between them (gold band) is the loop; the branches crossing in $E/N$ is the first-order energy kink.

In [ ]:
# ====================== 3 · PLOT + METRICS ======================
def aligned(fwd, bwd, key):
    """(h, y_fwd, y_bwd) on the fields common to both branches (shared grid)."""
    if fwd.get(key) is None or bwd.get(key) is None: return None
    hb = {round(h,4): v for h, v in zip(bwd["h"], bwd[key])}
    H, A, B = [], [], []
    for hi, av in zip(fwd["h"], fwd[key]):
        if round(hi,4) in hb: H.append(hi); A.append(av); B.append(hb[round(hi,4)])
    return np.array(H), np.array(A), np.array(B)

NROW = 3
fig, ax = plt.subplots(NROW, len(DATA), figsize=(7*len(DATA), 4.2*NROW), squeeze=False)
for j, (name, D) in enumerate(DATA.items()):
    fwd, bwd, f = D["fwd"], D["bwd"], D["field"]
    rows = MOVERS[f] + [("__E__", "$E/N$")]
    N = N_EDGES(fwd["L"]) if fwd else N_EDGES(L)
    for i, (key, ylab) in enumerate(rows):
        a = ax[i, j]
        if fwd is None or bwd is None:
            a.text(0.5, 0.5, "branch(es) missing", ha="center", transform=a.transAxes)
            if i == 0: a.set_title(name)
            continue
        yf = fwd["E"]/N if key == "__E__" else fwd.get(key)
        yb = bwd["E"]/N if key == "__E__" else bwd.get(key)
        if yf is None or yb is None:
            a.text(0.5, 0.5, f"no {key} (in-flight?)", ha="center", transform=a.transAxes); continue
        a.plot(fwd["h"], yf, "o-", color=BLUE, ms=5, lw=1.3, label="forward (h\u2191)")
        a.plot(bwd["h"], yb, "o-", color=RED,  ms=5, lw=1.3, label="backward (h\u2193)")
        a.set(xlabel=f"${f}$", ylabel=(ylab if key != '__E__' else '$E/N$'))
        # loop band + width on the magnetization row (row 0)
        title = name if i == 0 else ylab
        if i == 0:
            al = aligned(fwd, bwd, key)
            if al is not None:
                hh, mf, mb = al; m = np.abs(mf - mb) > LOOP_TOL
                loop_w = (hh[m].max() - hh[m].min()) if m.any() else 0.0
                if m.any(): a.axvspan(hh[m].min(), hh[m].max(), color="gold", alpha=0.15)
                title = f"{name}   [{D['order']}]   loop \u0394{f}={loop_w:.3f}"
        a.set_title(title, fontsize=11); a.legend(fontsize=8)
plt.tight_layout(); plt.show()
print("Loop width > 0 (branches separate) => first order.  Branches coincide => second order.")